In [ ]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)


In [ ]:
v1[0]

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
embed = Embedder()
q1 = "How does approximate nearest neighbor search work?"
v_query = embed.encode(q1)

doc = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

v_doc = embed.encode(doc["content"])

v_query.dot(v_doc)


In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
import numpy as np

embed = Embedder()
q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)

chunk_texts = [chunk["content"] for chunk in chunks]
v_chunks = embed.encode_batch(chunk_texts)

X = np.array(v_chunks)


In [ ]:
scores = X.dot(v1)

In [ ]:
idx = np.argmax(scores)
chunks[idx]["filename"]

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

q2 = "What metric do we use to evaluate a search engine?"
v2= embed.encode(q2)
vindex.search(v2)


In [ ]:
query = "How do I store vectors in PostgreSQL?"
v = embed.encode(query)
results = vindex.search(v, num_results=5)
results

In [ ]:
from minsearch import Index

index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)

index.fit(chunks)

In [ ]:
question = "How do I store vectors in PostgreSQL?"

search_results = index.search(
    question,
    boost_dict = {'content': 2.0},
    num_results=5
    )

search_results

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
vec_query = "How do I give the model access to tools?"
v_query = embed.encode(vec_query)
vector_results = vindex.search(v_query)
vector_results

In [ ]:
ind_question = "How do I give the model access to tools?"

text_results = index.search(
    ind_question,
    boost_dict = {'content': 2.0},
    )
text_results

In [ ]:
results = rrf([vector_results, text_results])
results